In [ ]:
import os
import json
import uuid
from pathlib import Path
from datetime import datetime

#making sure ROOT and keys are set
ROOT = Path(r"C:\Users\MAITHILI\Care_Companion")

os.environ["GROQ_API_KEY"] = "*****"
os.environ["USDA_API_KEY"] = "*****"

print("Imports done.")
print(f"ROOT exists: {ROOT.exists()}")

Imports done.
ROOT exists: True


In [2]:
#creating SQLite database and patient table

from sqlalchemy import (
    create_engine, Column, String, Integer, 
    Float, Text, DateTime
)
from sqlalchemy.orm import declarative_base, Session

Base = declarative_base()
DB_PATH = ROOT / "memory" / "sqlite" / "carecompanion.db"

class Patient(Base):
    __tablename__ = "patients"

    patient_id       = Column(String,  primary_key=True)
    name             = Column(String)
    age              = Column(Integer)
    created_at       = Column(String)
    last_seen        = Column(String)

    # Medical
    conditions       = Column(Text)   
    medications      = Column(Text)   
    allergies        = Column(Text)   
    recent_glucose   = Column(Float)
    recent_a1c       = Column(Float)
    recent_bmi       = Column(Float)

    # Lifestyle
    diet_preferences = Column(Text)   
    foods_to_avoid   = Column(Text)   
    activity_level   = Column(String)

    # Financial
    budget_monthly   = Column(Float)
    insurance        = Column(String)

    # Agent memory
    session_count    = Column(Integer, default=0)
    agent_notes      = Column(Text)   
    last_fda_check   = Column(String, nullable=True)

#creating the database
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)
Base.metadata.create_all(engine)

print(f"SQLite database created: {DB_PATH}")
print("Tables created: patients")

SQLite database created: C:\Users\MAITHILI\Care_Companion\memory\sqlite\carecompanion.db
Tables created: patients


In [3]:
from typing import Optional

def save_patient(patient: dict):
    """Save or update a patient profile in SQLite."""
    with Session(engine) as session:
        db_patient = Patient(
            patient_id       = patient["patient_id"],
            name             = patient["name"],
            age              = patient["age"],
            created_at       = patient["created_at"],
            last_seen        = datetime.now().isoformat(),
            conditions       = json.dumps(patient["conditions"]),
            medications      = json.dumps(patient["medications"]),
            allergies        = json.dumps(patient["allergies"]),
            recent_glucose   = patient["recent_glucose"],
            recent_a1c       = patient["recent_a1c"],
            recent_bmi       = patient["recent_bmi"],
            diet_preferences = json.dumps(patient["diet_preferences"]),
            foods_to_avoid   = json.dumps(patient["foods_to_avoid"]),
            activity_level   = patient["activity_level"],
            budget_monthly   = patient["budget_monthly"],
            insurance        = patient["insurance"],
            session_count    = patient["session_count"],
            agent_notes      = json.dumps(patient["agent_notes"]),
            last_fda_check   = patient["last_fda_check"],
        )
        session.merge(db_patient)
        session.commit()


def load_patient(patient_id: str) -> Optional[dict]:
    """Load a patient profile from SQLite. Returns dict or None."""
    with Session(engine) as session:
        row = session.get(Patient, patient_id)
        if row is None:
            return None
        return {
            "patient_id":       row.patient_id,
            "name":             row.name,
            "age":              row.age,
            "created_at":       row.created_at,
            "last_seen":        row.last_seen,
            "conditions":       json.loads(row.conditions),
            "medications":      json.loads(row.medications),
            "allergies":        json.loads(row.allergies),
            "recent_glucose":   row.recent_glucose,
            "recent_a1c":       row.recent_a1c,
            "recent_bmi":       row.recent_bmi,
            "diet_preferences": json.loads(row.diet_preferences),
            "foods_to_avoid":   json.loads(row.foods_to_avoid),
            "activity_level":   row.activity_level,
            "budget_monthly":   row.budget_monthly,
            "insurance":        row.insurance,
            "session_count":    row.session_count,
            "agent_notes":      json.loads(row.agent_notes),
            "last_fda_check":   row.last_fda_check,
        }


def update_patient_field(patient_id: str, field: str, value):
    """Update a single field for a patient."""
    with Session(engine) as session:
        row = session.get(Patient, patient_id)
        if row:
            if isinstance(value, (list, dict)):
                setattr(row, field, json.dumps(value))
            else:
                setattr(row, field, value)
            row.last_seen = datetime.now().isoformat()
            session.commit()


def list_all_patients() -> list:
    """Return a summary list of all patients in the database."""
    with Session(engine) as session:
        rows = session.query(Patient).all()
        return [
            {
                "patient_id":    r.patient_id,
                "name":          r.name,
                "age":           r.age,
                "session_count": r.session_count,
                "last_seen":     r.last_seen,
            }
            for r in rows
        ]

print("SQLite helper functions defined:")
print("  save_patient(patient)")
print("  load_patient(patient_id)")
print("  update_patient_field(patient_id, field, value)")
print("  list_all_patients()")

SQLite helper functions defined:
  save_patient(patient)
  load_patient(patient_id)
  update_patient_field(patient_id, field, value)
  list_all_patients()


In [4]:
#testing SQLite with example patient

# Creating a test patient
test_patient = {
    "patient_id":       "p001",
    "name":             "Sarah",
    "age":              50,
    "created_at":       datetime.now().isoformat(),
    "last_seen":        datetime.now().isoformat(),
    "conditions":       ["Type 2 Diabetes"],
    "medications":      [{"name": "Metformin", "dose": "500mg", 
                          "frequency": "twice daily"}],
    "allergies":        [],
    "recent_glucose":   148.0,
    "recent_a1c":       7.2,
    "recent_bmi":       33.6,
    "diet_preferences": ["low-carb"],
    "foods_to_avoid":   [],
    "activity_level":   "light",
    "budget_monthly":   150.0,
    "insurance":        "none",
    "session_count":    0,
    "agent_notes":      [],
    "last_fda_check":   None,
}

#savung
save_patient(test_patient)
print("Saved Sarah to SQLite")

#loading it back
loaded = load_patient("p001")
print(f"Loaded back: {loaded['name']}, age {loaded['age']}")
print(f"Medications: {loaded['medications']}")
print(f"Glucose:     {loaded['recent_glucose']}")

#updating
update_patient_field("p001", "recent_glucose", 132.0)
updated = load_patient("p001")
print(f"\nAfter glucose update: {updated['recent_glucose']} (was 148.0)")

#adding an agent node
notes = updated["agent_notes"]
notes.append("Patient mentioned eating white rice frequently — flagged for nutrition guidance")
update_patient_field("p001", "agent_notes", notes)
final = load_patient("p001")
print(f"Agent note saved: {final['agent_notes'][0][:60]}...")

Saved Sarah to SQLite
Loaded back: Sarah, age 50
Medications: [{'name': 'Metformin', 'dose': '500mg', 'frequency': 'twice daily'}]
Glucose:     148.0

After glucose update: 132.0 (was 148.0)
Agent note saved: Patient mentioned eating white rice frequently — flagged for...


In [5]:
#Loading 20 patients into SQLite

eval_path = ROOT / "evaluation" / "test_patients.json"
with open(eval_path, "r", encoding="utf-8") as f:
    test_patients = json.load(f)

for p in test_patients:
    save_patient(p)

all_patients = list_all_patients()
print(f"Loaded {len(all_patients)} patients into SQLite\n")
print(f"{'ID':6s} {'Name':10s} {'Age':4s} {'Sessions':8s} Last Seen")
print("-" * 55)
for p in all_patients:
    seen = p['last_seen'][:10]
    print(f"{p['patient_id']:6s} {p['name']:10s} {p['age']:4d} "
          f"{p['session_count']:8d} {seen}")

Loaded 20 patients into SQLite

ID     Name       Age  Sessions Last Seen
-------------------------------------------------------
p001   Sarah        50        0 2026-05-09
p002   James        32        0 2026-05-09
p003   Maria        33        0 2026-05-09
p004   David        26        0 2026-05-09
p005   Linda        53        0 2026-05-09
p006   Carlos       54        0 2026-05-09
p007   Emma         34        0 2026-05-09
p008   Michael      59        0 2026-05-09
p009   Priya        51        0 2026-05-09
p010   Robert       32        0 2026-05-09
p011   Angela       31        0 2026-05-09
p012   Kevin        31        0 2026-05-09
p013   Susan        32        0 2026-05-09
p014   Ahmed        41        0 2026-05-09
p015   Nicole       29        0 2026-05-09
p016   Thomas       51        0 2026-05-09
p017   Grace        41        0 2026-05-09
p018   Daniel       43        0 2026-05-09
p019   Patricia     28        0 2026-05-09
p020   Marcus       46        0 2026-05-09


In [6]:
import chromadb

chroma_client = chromadb.PersistentClient(
    path=str(ROOT / "memory" / "chroma_db")
)


conversations = chroma_client.get_or_create_collection(
    name="conversations",
    metadata={"hnsw:space": "cosine"}  
)

fda_alerts = chroma_client.get_or_create_collection(
    name="fda_alerts",
    metadata={"hnsw:space": "cosine"}
)

print("ChromaDB collections ready:")
print(f"  conversations: {conversations.count()} documents")
print(f"  fda_alerts:    {fda_alerts.count()} documents")

ChromaDB collections ready:
  conversations: 36 documents
  fda_alerts:    0 documents


In [7]:
def save_message(patient_id: str, role: str, content: str, session_id: str):
    """
    Save a single message to ChromaDB.
    role = 'user' or 'assistant'
    """
    doc_id = f"{patient_id}_{session_id}_{datetime.now().timestamp()}"
    
    conversations.add(
        documents=[content],
        metadatas=[{
            "patient_id": patient_id,
            "role":       role,
            "session_id": session_id,
            "timestamp":  datetime.now().isoformat(),
        }],
        ids=[doc_id]
    )


def get_relevant_history(patient_id: str, query: str, n: int = 5) -> list:
    """
    Retrieve the most semantically relevant past messages for a patient.
    This is how the agent 'remembers' things from past sessions.
    """
    results = conversations.query(
        query_texts=[query],
        n_results=min(n, conversations.count()) if conversations.count() > 0 else 1,
        where={"patient_id": patient_id}
    )
    
    if not results["documents"][0]:
        return []
    
    history = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        history.append({
            "role":      meta["role"],
            "content":   doc,
            "timestamp": meta["timestamp"],
            "session":   meta["session_id"],
        })
    return history


def get_full_session_history(patient_id: str, session_id: str) -> list:
    """Get all messages from a specific session in order."""
    results = conversations.get(
        where={
            "$and": [
                {"patient_id": {"$eq": patient_id}},
                {"session_id": {"$eq": session_id}}
            ]
        }
    )
    
    if not results["documents"]:
        return []
    
    #sorting by timestamp
    messages = [
        {"role": m["role"], "content": d, "timestamp": m["timestamp"]}
        for d, m in zip(results["documents"], results["metadatas"])
    ]
    return sorted(messages, key=lambda x: x["timestamp"])


print("ChromaDB helper functions defined:")
print("  save_message(patient_id, role, content, session_id)")
print("  get_relevant_history(patient_id, query, n)")
print("  get_full_session_history(patient_id, session_id)")

ChromaDB helper functions defined:
  save_message(patient_id, role, content, session_id)
  get_relevant_history(patient_id, query, n)
  get_full_session_history(patient_id, session_id)


In [8]:
#example
session_1 = "session_001"

#session 1 messages
convo_1 = [
    ("user",      "Hi, I was just diagnosed with Type 2 diabetes and I'm feeling overwhelmed."),
    ("assistant", "I understand this can feel overwhelming. You're taking the right step by seeking information. Can you tell me what medications you've been prescribed?"),
    ("user",      "I'm on Metformin 500mg twice a day. I'm also worried about the cost."),
    ("assistant", "Metformin is actually one of the most affordable diabetes medications. Generic versions cost as little as $4-10 per month at most pharmacies."),
    ("user",      "That's a relief. I've also been eating a lot of white rice. Is that okay?"),
    ("assistant", "White rice has a high glycemic index and can spike blood sugar significantly. Given your diabetes, I'd suggest switching to cauliflower rice or brown rice as alternatives."),
]

for role, content in convo_1:
    save_message("p001", role, content, session_1)

print(f"Saved {len(convo_1)} messages from Session 1\n")

#now diff topic
session_2 = "session_002"
convo_2 = [
    ("user",      "I had my blood test, glucose is 132 now, down from 148."),
    ("assistant", "That's great progress! A drop from 148 to 132 mg/dL shows your Metformin and dietary changes are working."),
    ("user",      "I've been avoiding white rice like you suggested."),
    ("assistant", "Excellent. Reducing high-glycemic foods is one of the most effective ways to manage Type 2 diabetes alongside medication."),
]

for role, content in convo_2:
    save_message("p001", role, content, session_2)

print(f"Saved {len(convo_2)} messages from Session 2\n")
print(f"Total messages in ChromaDB: {conversations.count()}")

Saved 6 messages from Session 1

Saved 4 messages from Session 2

Total messages in ChromaDB: 46


In [9]:
print("Testing semantic memory retrieval for Sarah (p001)")
print("=" * 55)

queries = [
    "what did the patient say about food?",
    "what are the patient's medication costs?",
    "how has glucose changed over time?",
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 45)
    history = get_relevant_history("p001", query, n=2)
    for msg in history:
        role    = msg["role"].upper()
        content = msg["content"][:80]
        session = msg["session"]
        print(f"  [{role} | {session}]: {content}...")

Testing semantic memory retrieval for Sarah (p001)

Query: 'what did the patient say about food?'
---------------------------------------------
  [ASSISTANT | session_002]: Excellent. Reducing high-glycemic foods is one of the most effective ways to man...
  [ASSISTANT | session_002]: Excellent. Reducing high-glycemic foods is one of the most effective ways to man...

Query: 'what are the patient's medication costs?'
---------------------------------------------
  [USER | session_20260507_143753]: I'm worried about the cost of my medications, they're getting expensive...
  [ASSISTANT | session_001]: Metformin is actually one of the most affordable diabetes medications. Generic v...

Query: 'how has glucose changed over time?'
---------------------------------------------
  [USER | session_002]: I had my blood test, glucose is 132 now, down from 148....
  [USER | session_002]: I had my blood test, glucose is 132 now, down from 148....


In [10]:
class MemoryManager:
    """
    Central memory interface for the CareCompanion agent.
    
    Combines SQLite (structured facts) + ChromaDB (conversation history)
    into a single clean interface the agent calls.
    """

    def __init__(self, patient_id: str):
        self.patient_id = patient_id
        self.session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.profile    = load_patient(patient_id)

        if self.profile is None:
            raise ValueError(f"Patient {patient_id} not found in database.")

    
        self.profile["session_count"] += 1
        update_patient_field(patient_id, "session_count", 
                             self.profile["session_count"])
        
        print(f"Session started for {self.profile['name']} "
              f"(session #{self.profile['session_count']})")

    def remember(self, role: str, content: str):
        """Save a message to conversation memory."""
        save_message(self.patient_id, role, content, self.session_id)

    def recall(self, query: str, n: int = 3) -> list:
        """Retrieve relevant past messages given a query."""
        return get_relevant_history(self.patient_id, query, n)

    def get_profile(self) -> dict:
        """Get the current patient profile."""
        return load_patient(self.patient_id)

    def update_profile(self, field: str, value):
        """Update a specific field in the patient profile."""
        update_patient_field(self.patient_id, field, value)
        self.profile = load_patient(self.patient_id)

    def add_note(self, note: str):
        """Add an agent observation note to the patient profile."""
        profile = self.get_profile()
        notes   = profile["agent_notes"]
        notes.append(f"[{datetime.now().strftime('%Y-%m-%d')}] {note}")
        self.update_profile("agent_notes", notes)

    def get_context_summary(self) -> str:
        """
        Build a structured context string for the LLM.
        This is injected into every agent prompt so the LLM
        always knows who it's talking to.
        """
        p    = self.get_profile()
        meds = ", ".join(
            f"{m['name']} {m['dose']} {m['frequency']}" 
            for m in p["medications"]
        )
        notes = "\n".join(p["agent_notes"][-3:]) if p["agent_notes"] else "None"

        return f"""
PATIENT PROFILE:
- Name: {p['name']}, Age: {p['age']}
- Conditions: {', '.join(p['conditions'])}
- Medications: {meds}
- Recent Glucose: {p['recent_glucose']} mg/dL
- Recent A1c: {p['recent_a1c']}%
- BMI: {p['recent_bmi']}
- Diet: {', '.join(p['diet_preferences'])}
- Activity: {p['activity_level']}
- Monthly Budget: ${p['budget_monthly']}
- Insurance: {p['insurance']}
- Sessions with agent: {p['session_count']}
- Agent notes: 
{notes}
""".strip()


print("MemoryManager class defined.")

MemoryManager class defined.


In [11]:
#starting a session for Sarah
memory = MemoryManager("p001")


print("\nContext summary the agent sees:")
print("=" * 55)
print(memory.get_context_summary())

memory.add_note("Patient is price-sensitive, prefers generic medications")
memory.add_note("Avoids white rice since session 1, glucose improving")

# to check notes were saved
profile = memory.get_profile()
print("\nAgent notes saved:")
for note in profile["agent_notes"]:
    print(f"  {note}")

#recall
print("\nRecalling relevant history for 'food and diet':")
history = memory.recall("food and diet", n=2)
for msg in history:
    print(f"  [{msg['role'].upper()}]: {msg['content'][:70]}...")

Session started for Sarah (session #1)

Context summary the agent sees:
PATIENT PROFILE:
- Name: Sarah, Age: 50
- Conditions: Type 2 Diabetes
- Medications: Metformin 500mg twice daily
- Recent Glucose: 148.0 mg/dL
- Recent A1c: 7.3%
- BMI: 33.6
- Diet: low-carb
- Activity: sedentary
- Monthly Budget: $200.0
- Insurance: none
- Sessions with agent: 1
- Agent notes: 
None

Agent notes saved:
  [2026-05-09] Patient is price-sensitive, prefers generic medications
  [2026-05-09] Avoids white rice since session 1, glucose improving

Recalling relevant history for 'food and diet':
  [ASSISTANT]: Excellent. Reducing high-glycemic foods is one of the most effective w...
  [ASSISTANT]: Excellent. Reducing high-glycemic foods is one of the most effective w...
